# A股主板低位筹码峰日线筛选器
收盘后或次日开盘前运行。代码来自 GitHub，缓存、状态和每日结果保存在 Google Drive。

In [ ]:
REPO_URL = 'https://github.com/morewnutri/a-share-screener.git'
BRANCH = 'main'
CONFIG_PATH = 'config/default.yaml'  # 高召回对照可改为 config/high_recall.yaml
DRIVE_DATA_DIR = '/content/drive/MyDrive/a_share_screener_data'
SKIP_FUND_FLOW = False  # 资金接口异常时改为 True，先发布纯形态结果
RUN_BACKTEST = False
BACKTEST_START = '2024-01-01'
BACKTEST_END = '2025-12-31'
REFERENCE_NAMES = [
    '中天科技', '远东股份', '兴发集团', '云南锗业', '超声电子', '京泉华',
    '通鼎互联', '福晶科技', '杭电股份', '亨通光电', '烽火通信', '中京电子',
    '博杰股份', '沃格光电', '红星发展', '翔鹭钨业', '新洁能', '赛腾股份',
    '伊戈尔', '华宏科技', '再升科技', '通达股份', '永鼎股份', '株冶集团',
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pathlib
import subprocess
import sys

if 'YOUR_NAME' in REPO_URL:
    raise ValueError('请先把 REPO_URL 改成你的 GitHub 仓库地址')

repo_dir = pathlib.Path('/content/ashare-daily-scanner')
if (repo_dir / '.git').exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir)], check=True)
os.chdir(repo_dir)
print(repo_dir)
subprocess.run([sys.executable, '-m', 'ashare_scanner', '--version'], check=True)

In [ ]:
command = [
    sys.executable, '-m', 'ashare_scanner',
    '--config', CONFIG_PATH,
    '--data-dir', DRIVE_DATA_DIR,
    'run', '--print-top', '30',
]
if SKIP_FUND_FLOW:
    command.append('--no-fund-flow')
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import display

latest = json.loads((pathlib.Path(DRIVE_DATA_DIR) / 'latest_run.json').read_text(encoding='utf-8'))
run_dir = pathlib.Path(latest['run_dir'])
report = json.loads((run_dir / 'coverage_report.json').read_text(encoding='utf-8'))
print('信号数量:', json.dumps(report['signals'], ensure_ascii=False))
print('诊断:', report['screening']['assessment'])
print('筹码口径:', report['data_policy']['chip_distribution'], '（非账户级持仓）')
print('资金排序:', json.dumps(report.get('fund_flow', {}), ensure_ascii=False))

labels = {
    'chip_base_ready': '低位横盘+筹码峰（待启动）',
    'chip_base_launch': '低位横盘+筹码峰（刚启动）',
    'chip_base_rebound': '横盘后反弹（启动确认）',
}
result_columns = [
    'rank', 'code', 'name', 'close', 'pct_chg', 'final_selection_score',
    'chip_base_ready_score', 'chip_base_launch_score', 'chip_base_rebound_score',
    'fund_flow_strength_score', 'institutional_dominance_score',
    'retail_pressure_index', 'participant_structure_label',
    'institutional_favorable_day_ratio_20_pct',
    'selection_evidence_coverage_pct', 'fund_flow_rank_reason',
    'main_net_inflow_3d_yi', 'main_net_inflow_5d_yi',
    'main_net_inflow_10d_yi', 'main_net_inflow_20d_yi',
    'adaptive_base_window', 'adaptive_base_offset',
    'adaptive_base_drawdown_120_pct', 'adaptive_base_width_pct',
    'adaptive_base_return_pct', 'distance_from_adaptive_base_high_pct',
    'chip_peak_price', 'chip_peak_distance_pct',
    'chip_peak_band_share_pct', 'chip_70_width_pct',
    'chip_low_zone_share_pct', 'chip_overhead_ratio_pct', 'return_5d_pct',
]
for signal, label in labels.items():
    result = pd.read_csv(run_dir / f'{signal}_all.csv', dtype={'code': str})
    print(f'\n[{label}] 共 {len(result)} 只')
    if result.empty:
        print('无符合条件股票；CSV 只有表头属于正常结果。')
    else:
        display(result[[c for c in result_columns if c in result.columns]].head(100))

print('\n当前观察池')
display(pd.read_csv(run_dir / 'watchlist_active.csv', dtype={'code': str}).head(100))

In [ ]:
# 参考样本由 CLI 优先抓取并审计，不参与任何评分或筛选。
audit = pd.read_csv(run_dir / 'reference_examples_audit.csv', dtype={'code': str})
ready = int(pd.to_numeric(audit['indicators_ready'], errors='coerce').fillna(0).sum())
print(f'参考样本指标可用 {ready}/{len(audit)} 只')
audit_columns = [
    'code', 'name', 'fetch_status', 'last_date', 'error',
    'chip_base_ready', 'chip_base_launch', 'chip_base_rebound',
    'closest_signal', 'failed_at', 'final_selection_score',
    'chip_base_ready_score', 'chip_base_launch_score', 'chip_base_rebound_score',
    'fund_flow_strength_score', 'institutional_dominance_score',
    'retail_pressure_index', 'participant_structure_label', 'fund_flow_rank_reason',
    'main_net_inflow_3d_yi', 'main_net_inflow_5d_yi',
    'main_net_inflow_10d_yi', 'main_net_inflow_20d_yi',
    'adaptive_base_window', 'adaptive_base_offset',
    'adaptive_base_drawdown_120_pct', 'adaptive_base_width_pct',
    'adaptive_base_return_pct', 'distance_from_adaptive_base_high_pct',
    'chip_peak_price', 'chip_peak_distance_pct',
    'chip_peak_band_share_pct', 'chip_70_width_pct', 'chip_low_zone_share_pct',
    'chip_peak_position', 'adaptive_trend_votes',
    'return_5d_pct', 'return_10d_pct',
]
display(audit[[c for c in audit_columns if c in audit.columns]].sort_values('name'))

In [ ]:
if RUN_BACKTEST:
    subprocess.run([
        sys.executable, '-m', 'ashare_scanner',
        '--config', CONFIG_PATH, '--data-dir', DRIVE_DATA_DIR,
        'backtest', '--start', BACKTEST_START, '--end', BACKTEST_END,
    ], check=True)